# Evaluation of the Inaccuracy class

This notebook demonstrates inaccuracy scores and analysis reports for fitted models and explicit predictions.


In [ ]:
import numpy as np

import matplotlib.pyplot as plt
from matplotlib.pylab import rcParams

from sklearn.datasets import load_digits, make_classification
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from sklearn.tree import DecisionTreeClassifier

from mnplib.inaccuracy import Inaccuracy, inaccuracy_predictions


In [ ]:
rcParams['figure.figsize'] = 10, 5

## Inaccuracy of a Model

The following example shows how inaccuracy can be used to evaluate a trained model.


In [ ]:
X, y = load_digits(return_X_y=True)
tree = DecisionTreeClassifier(min_samples_leaf=5, random_state=42)
tree.fit(X, y)

In [ ]:
inacc = Inaccuracy(y_type="categorical")
inacc.fit(X, y)
inacc.inaccuracy_model(tree)

We can also work directly with predictions instead of using the model object. This is useful for models that do not follow the scikit-learn estimator interface.


In [ ]:
pred = tree.predict(X)
inacc.inaccuracy_predictions(pred)

The same prediction-level computation is also available as a functional interface.


In [ ]:
inaccuracy_predictions(pred, y_type='categorical', y=y)

## Prediction Analysis
The report contains code lengths in bits, descriptive joint-state sparsity statistics,
and classification accuracy. Numeric targets instead include MAE and RMSE in target
units. These conventional errors are distinct from information-based inaccuracy.
`pformat(report)` returns a formatted numerical dictionary; the dictionary retains all diagnostics.

In [ ]:
from mnplib.inaccuracy import prediction_analysis, model_analysis
from pprint import pformat

report = inacc.model_analysis(tree)
assert report == inacc.prediction_analysis(pred)
assert report == prediction_analysis(pred, y=y)
assert report == model_analysis(tree, X=X, y=y)
assert report["inaccuracy"] == inacc.inaccuracy_predictions(pred)
print(pformat(report))

In [ ]:
numeric_y = np.array([0., 1., 2., 3.])
numeric_predictions = np.array([0., 2., 1., 5.])
numeric_report = Inaccuracy(y_type="numeric").fit_y(numeric_y).prediction_analysis(numeric_predictions)
assert np.isclose(numeric_report["mae"], 1.)
assert np.isclose(numeric_report["rmse"], np.sqrt(1.5))
print(pformat(numeric_report))

## Compare score with inaccuracy

Train a collection of decision-tree classifiers with different maximum depths, compute classical classification error and inaccuracy, and compare the results.


In [ ]:
X, y = load_digits(return_X_y=True)
inacc = Inaccuracy(y_type="categorical")
inacc.fit(X, y)

In [ ]:
classical_errors = []
inaccuracies = []

for depth in range(1, 20):
    tree = DecisionTreeClassifier(max_depth=depth, random_state=42)
    tree.fit(X, y)
    
    classical_errors.append(1.0 - tree.score(X, y))
    inaccuracies.append(inacc.inaccuracy_model(tree))

In [ ]:
plt.plot(range(1, 20), classical_errors, label="1 - accuracy")
plt.plot(range(1, 20), inaccuracies, label="Inaccuracy")
plt.ylabel("Error")
plt.xlabel("Tree depth")
plt.legend()
plt.show()

## Adding errors

Study the behavior of accuracy error and inaccuracy when we introduce errors in the dataset.


In [ ]:
tree = DecisionTreeClassifier(min_samples_leaf=5, random_state=42)
tree.fit(X, y)

inacc.fit(X, y)
inacc.inaccuracy_model(tree)

In [ ]:
1.0 - tree.score(X, y)

Now repeat the same wrong example one hundred times.


In [ ]:
X2 = X.copy()
y2 = y.copy()

for _ in range(100):
    X2 = np.append(X2, [X[0]], axis=0)
    y2 = np.append(y2, (y[0] + 1) % 10)

In [ ]:
inacc.fit(X2, y2)
inacc.inaccuracy_model(tree)

In [ ]:
1.0 - tree.score(X2, y2)

The theory of nescience states that repeating the same error one hundred times is less informative than making one hundred different errors. Now compare this with one hundred different wrong examples.


In [ ]:
rng = np.random.default_rng(42)

X3 = X.copy()
y3 = y.copy()

for _ in range(100):
    index = rng.integers(X.shape[0])
    X3 = np.append(X3, [X[index]], axis=0)
    y3 = np.append(y3, (y[index] + 1) % 10)

In [ ]:
inacc.fit(X3, y3)
inacc.inaccuracy_model(tree)

In [ ]:
1.0 - tree.score(X3, y3)

Making one hundred different errors is worse than making the same error one hundred times, but classical accuracy error is less sensitive to this distinction.


## Imbalanced dataset

Study the behaviour of score and inaccuracy in a highly imbalanced dataset.

In [ ]:
leaf_sizes = []
classical_errors = []
inaccuracies = []

inaccuracy = Inaccuracy(y_type="categorical")

X, y = make_classification(
    n_samples=1000,
    n_features=2,
    n_informative=2,
    n_redundant=0,
    class_sep=2,
    flip_y=0,
    weights=[0.95, 0.05],
    random_state=42,
)

inaccuracy.fit(X, y)

for leaf_size in range(1, 100):
    tree = DecisionTreeClassifier(min_samples_leaf=leaf_size, random_state=42)
    tree.fit(X, y)

    leaf_sizes.append(leaf_size)
    classical_errors.append(1.0 - tree.score(X, y))
    inaccuracies.append(inaccuracy.inaccuracy_model(tree))


In [ ]:
plt.plot(leaf_sizes, classical_errors, label="1 - accuracy")
plt.plot(leaf_sizes, inaccuracies, label="Inaccuracy")
plt.title("Imbalanced binary classification")
plt.ylabel("Error")
plt.xlabel("Minimum leaf size")
plt.legend(loc="best")
plt.show()

As we can see, classical accuracy error can be misleading on highly imbalanced datasets. Inaccuracy provides a complementary information-theoretic view of the predictions.


## A comparison to precision, recall, F1, ROC AUC

Standard classification metrics evaluate specific operational aspects of prediction, whereas inaccuracy evaluates the amount of information about the target preserved by the predictions. Therefore, inaccuracy can distinguish cases that accuracy, precision, recall, F1, and ROC AUC collapse into the same value.

In [ ]:
rng = np.random.default_rng(42)

# Balanced 5-class target.
n_per_class = 100
y = np.repeat(np.arange(5), n_per_class)
X_dummy = np.zeros((len(y), 1))

metric = Inaccuracy(y_type="categorical")
metric.fit(X_dummy, y)

Prediction 1: systematic cyclic permutation for 40% of samples.


In [ ]:
pred = y.copy()
idx = rng.choice(len(y), size=int(0.4 * len(y)), replace=False)
pred[idx] = (pred[idx] + 1) % 5

In [ ]:
print("Accuracy:", accuracy_score(y, pred))
print("Macro precision:", precision_score(y, pred, average="macro"))
print("Macro recall:", recall_score(y, pred, average="macro"))
print("Macro F1:", f1_score(y, pred, average="macro"))
print("Inaccuracy:", metric.inaccuracy_predictions(pred))

Prediction 2: random wrong labels for the same samples.


In [ ]:
pred = y.copy()
for i in idx:
    alternatives = [c for c in range(5) if c != y[i]]
    pred[i] = rng.choice(alternatives)

In [ ]:
print("Accuracy:", accuracy_score(y, pred))
print("Macro precision:", precision_score(y, pred, average="macro"))
print("Macro recall:", recall_score(y, pred, average="macro"))
print("Macro F1:", f1_score(y, pred, average="macro"))
print("Inaccuracy:", metric.inaccuracy_predictions(pred))